In [ ]:
from steer_core.Data.DataManager import DataManager
import steer_opencell_design as ocd

In [ ]:
# User Inputs

####################

table_name = 'cell_references' #### change this to cell_teardowns for teardowns
chemistry = 'Li/Li+'   #### change to 'Li/Li+' for lithium-ion cells
cell_name = "NMC811 Stacked Prismatic Cell"

#####################

In [ ]:
# set some standard materials

conductive_additive = ocd.ConductiveAdditive.from_database("Super P")
binder = ocd.Binder.from_database("PVDF")
insulation = ocd.InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = ocd.SeparatorMaterial.from_database('Polyethylene')
tape_material = ocd.TapeMaterial.from_database("Kapton")
prismatic_material = ocd.PrismaticContainerMaterial.from_database("Steel")

In [ ]:
# Create the cathode

cathode_current_collector_material = ocd.CurrentCollectorMaterial.from_database('Aluminum')

cathode_current_collector=ocd.PunchedCurrentCollector(
    material=cathode_current_collector_material,
    width=300,
    height=280,
    tab_height=30,
    tab_position=230,
    tab_width=80,
    thickness=10,
    insulation_width=2
)

cathode_active_material = ocd.CathodeMaterial.from_database("NMC811")

cathode_formulation = ocd.CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = ocd.Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=3.1,
    mass_loading=14,
    insulation_material=insulation,
    insulation_thickness=3
)


In [ ]:
# Create the anode

cathode_current_collector_material = ocd.CurrentCollectorMaterial.from_database("Copper")

anode_current_collector = ocd.PunchedCurrentCollector(
    material=cathode_current_collector_material,
    width=302,
    height=280,
    tab_height=30,
    tab_position=230,
    tab_width=80,
    thickness=10
)

anode_active_material = ocd.AnodeMaterial.from_database("Synthetic Graphite")

anode_formulation = ocd.AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = ocd.Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=9
)

In [ ]:
# create the layup 

separator = ocd.Separator(
    material=separator_material, 
    thickness=12,
    width = 290,
)

my_layup = ocd.ZFoldMonoLayer(
    cathode=my_cathode,
    anode=my_anode,
    separator=separator,
    electrode_orientation='transverse'
)

my_layup.get_top_down_view().show()


In [ ]:
# create the stack assembly

my_stack = ocd.ZFoldStack(
    layup=my_layup,
    n_layers=40,
    additional_separator_wraps=3
)

# looks best in safari
my_stack.get_side_view().show()

In [ ]:
# make the electrolyte

my_electrolyte = ocd.Electrolyte(
    name="1M NaPF6 in EC:PC:DMC (1:1:1 wt%)",
    density=1.2,
    specific_cost=10,
    color="#FF9D00"
)


In [ ]:
# make the encapsulation

cathode_terminal_connector = ocd.PrismaticTerminalConnector(
    material=prismatic_material,
    thickness=2,
    width=40,
    length=120
)

anode_terminal_connector = ocd.PrismaticTerminalConnector(
    material=prismatic_material,
    thickness=2,
    width=40,
    length=120
)

lid_assembly = ocd.PrismaticLidAssembly(
    material=prismatic_material,
    thickness=8,
)

cannister = ocd.PrismaticCannister(
    material=prismatic_material,
    width=295,
    length=52,
    height=315,
    wall_thickness=1
)

my_encapsulation = ocd.PrismaticEncapsulation(
    cannister=cannister,
    cathode_terminal_connector=cathode_terminal_connector,
    anode_terminal_connector=anode_terminal_connector,
    lid_assembly=lid_assembly,
    connector_orientation='transverse'
)

In [ ]:
# make the cell

cell = ocd.PrismaticCell(
    reference_electrode_assembly=my_stack,
    electrolyte=my_electrolyte,
    electrolyte_overfill=0.1,
    encapsulation=my_encapsulation,
    n_electrode_assembly=4,
    clipped_tab_length=4,
    name=cell_name
)

# looks better in safari
cell.get_side_view().show()
cell.get_top_down_view().show()

In [ ]:
cell.plot_mass_breakdown(width=800, height=800)

In [ ]:
cell.plot_cost_breakdown(width=800, height=800)

In [ ]:
cell.get_capacity_plot(width=1300, height=800).show()

In [ ]:
print(f"Cost ($): {cell.cost}")
print(f"Mass (g): {cell.mass}")
print(f"Energy Density (Wh/L): {cell.volumetric_energy}")
print(f"Energy (Wh): {cell.energy}")
print(f"Energy Density (Wh/kg): {cell.specific_energy}")
print(f"Normalized Cost ($/kWh): {cell.cost_per_energy}")

In [ ]:
import pandas as pd
import datetime as dt
from steer_opencell_design import __version__


db = DataManager()

db.remove_data(table_name=table_name, condition=f"name = '{cell.name}'")

internal_construction = cell.reference_electrode_assembly.__class__.__name__
form_factor = cell.__class__.__name__

# insert the cell into the database
db.insert_data(table_name=table_name, data=pd.DataFrame({
    'name': [cell.name],
    'object': [cell.serialize()],
    'form_factor': [form_factor],
    'internal_construction': [internal_construction],
    'date_created': [dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'chemistry': [chemistry]
}))

db.get_data(table_name)